# Geospatial Clustering for Competitive Analysis

This notebook builds a location-based clustering pipeline and derives competitive benchmarks for Airbnb listings.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import joblib

In [ ]:
project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent
data_path = project_root / "data" / "processed" / "main" / "feature_engineered.csv"
models_dir = project_root / "models"
models_dir.mkdir(exist_ok=True)

df = pd.read_csv(data_path)
print(f"Loaded: {data_path}")
print(f"Shape: {df.shape}")

In [ ]:
required_cols = [
    "latitude",
    "longitude",
    "ttm_avg_rate",
    "avg_rating",
    "num_reviews",
    "total_amenities",
    "luxury_score",
    "rarity_score",
    "comfort_to_total_ratio",
]

optional_cols = [
    "basic_count",
    "comfort_count",
    "kitchen_count",
    "safety_count",
    "outdoor_count",
    "family_count",
    "services_count",
    "has_pool",
    "has_hot_tub",
    "has_gym",
    "has_beach_access",
    "has_dedicated_workspace",
    "has_pets_allowed",
    "has_free_parking_on_premises",
    "has_air_conditioning",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

available_optional = [c for c in optional_cols if c in df.columns]
cluster_df = df[required_cols + available_optional].copy()
cluster_df = cluster_df.dropna(subset=["latitude", "longitude", "ttm_avg_rate"])

# Stored target is log-space; convert for business-level competitive benchmarks.
cluster_df["ttm_avg_rate_business"] = np.maximum(np.expm1(cluster_df["ttm_avg_rate"].astype(float)), 0.0)

# Clustering-friendly engineered features from amenities and engagement.
cluster_df["reviews_log"] = np.log1p(np.clip(cluster_df["num_reviews"], a_min=0, a_max=None))
cluster_df["amenity_premium_index"] = cluster_df["luxury_score"].fillna(0) + cluster_df["rarity_score"].fillna(0)

count_cols = [c for c in ["basic_count", "comfort_count", "kitchen_count", "safety_count", "outdoor_count", "family_count", "services_count"] if c in cluster_df.columns]
if count_cols:
    cluster_df["amenity_utility_index"] = cluster_df[count_cols].sum(axis=1)
else:
    cluster_df["amenity_utility_index"] = cluster_df["total_amenities"].fillna(0)

flag_cols = [c for c in [
    "has_pool",
    "has_hot_tub",
    "has_gym",
    "has_beach_access",
    "has_dedicated_workspace",
    "has_pets_allowed",
    "has_free_parking_on_premises",
    "has_air_conditioning",
] if c in cluster_df.columns]
if flag_cols:
    cluster_df["amenity_binary_sum"] = cluster_df[flag_cols].sum(axis=1)
else:
    cluster_df["amenity_binary_sum"] = 0

cluster_df = cluster_df.replace([np.inf, -np.inf], np.nan).dropna()

print("Using columns for clustering + profiling:")
print(cluster_df.columns.tolist())
print(cluster_df.describe().T)

In [ ]:
X_geo = cluster_df[["latitude", "longitude"]].to_numpy()

candidate_k = list(range(5, 16))
scores = []
for k in candidate_k:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_geo)
    sil = silhouette_score(X_geo, labels)
    scores.append((k, sil))

scores_df = pd.DataFrame(scores, columns=["k", "silhouette"]).sort_values("silhouette", ascending=False)
best_k = int(scores_df.iloc[0]["k"])
print(scores_df.head())
print(f"Selected k: {best_k}")

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_df["cluster_id"] = kmeans.fit_predict(X_geo)

cluster_profile = (
    cluster_df.groupby("cluster_id")
    .agg(
        listings=("cluster_id", "size"),
        median_rate=("ttm_avg_rate_business", "median"),
        median_rating=("avg_rating", "median"),
        median_reviews=("num_reviews", "median"),
        median_total_amenities=("total_amenities", "median"),
        median_luxury_score=("luxury_score", "median"),
        median_rarity_score=("rarity_score", "median"),
        median_amenity_utility=("amenity_utility_index", "median"),
        median_amenity_premium=("amenity_premium_index", "median"),
        median_binary_flags=("amenity_binary_sum", "median"),
        lat_center=("latitude", "mean"),
        lon_center=("longitude", "mean"),
    )
    .reset_index()
    .sort_values("median_rate")
    .reset_index(drop=True)
)

cluster_profile

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=cluster_df.sample(min(len(cluster_df), 10000), random_state=42),
    x="longitude",
    y="latitude",
    hue="cluster_id",
    palette="tab20",
    s=10,
    alpha=0.7,
)
plt.title("Geospatial Listing Clusters")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
cluster_output_path = models_dir / "competitive_cluster_profile.csv"
kmeans_path = models_dir / "competitive_geo_kmeans.pkl"
clustered_rows_path = models_dir / "competitive_clustered_listings_sample.csv"

cluster_profile.to_csv(cluster_output_path, index=False)
joblib.dump(kmeans, kmeans_path)

# Save a light sample for debugging/QA in API and dashboard development.
cols_for_sample = [
    "latitude",
    "longitude",
    "cluster_id",
    "ttm_avg_rate_business",
    "avg_rating",
    "num_reviews",
    "total_amenities",
    "amenity_premium_index",
    "amenity_utility_index",
]
existing_sample_cols = [c for c in cols_for_sample if c in cluster_df.columns]
cluster_df[existing_sample_cols].sample(min(len(cluster_df), 10000), random_state=42).to_csv(
    clustered_rows_path, index=False
)

print(f"Saved cluster profile: {cluster_output_path}")
print(f"Saved KMeans model: {kmeans_path}")
print(f"Saved clustered sample: {clustered_rows_path}")